In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time
from openai import OpenAI
from google.colab import userdata

In [ ]:
InputCsv="results_fetch_math_full.csv"
dataframe = pd.read_csv(InputCsv)
dataframe.head()

In [ ]:
def fetch_html_body_content(html_url):

    try:
        r = requests.get(html_url, timeout=30)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
    except Exception as e:
        print("⚠ HTML 请求失败:", e)
        return None, "html_fetch_failed"

    body = soup.find("body")
    if not body:
        return None, "no_body"

    lines = [line.strip() for line in body.stripped_strings if line.strip()]
    text = "\n".join(lines)

    if not text:
        return None, "empty_text"

    # 移除 Abstract
    text = re.sub(
        r'Abstract[:\s].*?(?=(Introduction|1\.\sIntroduction))',
        '',
        text,
        flags=re.S|re.I
    )

    # 截断 Acknowledgment
    ack_patterns = [
        r'Acknowledgment',
        r'Acknowledgements',
        r'ACKNOWLEDGMENT',
        r'ACKNOWLEDGEMENTS'
    ]

    for pat in ack_patterns:
        match = re.search(pat, text, re.I)
        if match:
            text = text[:match.start()]
            break

    text = "\n".join(line for line in text.splitlines() if line.strip())

    return text, "html_success"

In [ ]:
client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
EMPTY_TEXT = ""

def rewrite_with_gpt(article_text):

    prompt = f"""
You are a professional science popularization writer.

Task:
Rewrite the following scientific paper content into a clear and accessible explanation.

Requirements:
- Audience: general public with high-school background
- Length: 150–200 words
- Focus on the main idea, method, and contribution
- Ignore appendices, proofs, and supplementary discussions
- Avoid equations and heavy jargon

Paper content (main body only):
{article_text[:8000]}
"""

    try:
        response = client.responses.create(
            model="gpt-5.2",
            input=prompt
        )

        return response.output_text

    except Exception as e:
        print("⚠ GPT 调用异常:", e)
        return EMPTY_TEXT

In [ ]:
BATCH_SIZE = 5  # 你可以改大一点，比如 10 或 20

if __name__ == "__main__":

    dataframe = pd.read_csv(InputCsv)  # 必须有 html_url 列

    gpt_outputs = []
    content_source = []

    total = len(dataframe)

    for batch_start in range(0, total, BATCH_SIZE):
        batch_end = min(batch_start + BATCH_SIZE, total)

        print(f"\nProcessing batch {batch_start//BATCH_SIZE + 1} "
              f"({batch_start+1}-{batch_end})")

        for row_index in range(batch_start, batch_end):

            html_url = dataframe.loc[row_index, "html_url"]
            print(f"  → Article {row_index+1}: {html_url}")

            # -------- 抓 HTML 正文 --------
            article_text, status = fetch_html_body_content(html_url)
            content_source.append(status)

            if not article_text:
                gpt_outputs.append(EMPTY_TEXT)
                print("     ✗ HTML scrape failed")
                continue

            print("     ✓ HTML scrape success → Sending to GPT...")

            # -------- 发送 GPT --------
            gpt_text = rewrite_with_gpt(article_text)
            gpt_outputs.append(gpt_text)

            print("     ✓ GPT done")

    # ===============================
    # Save results
    # ===============================
    dataframe["gpt_prompt"] = gpt_outputs
    dataframe["content_source"] = content_source

    dataframe.to_csv(
        "results_html_to_gpt_output.csv",
        index=False
    )

    print("\n✅ Saved to results_math_gpt.csv")